# Telugu LoRA vs QLoRA — full training run (Stage 7)

Runs **one** full training (1 epoch, ~6–7 h) top to bottom with no manual steps, so it can run as a
background **"Save & Run All (Commit)"** job — the browser can be closed.

Settings: **Accelerator = GPU T4** · **Internet = On** · Secret **HF_TOKEN** attached.
Set `RUN` below, then **Save Version → Save & Run All (Commit)**. Do `lora` first, then `qlora`.

Adapters, checkpoints (`last-checkpoint/`) and TensorBoard logs are pushed to private HF repos
`thirumalreddy0172/qwen3-4b-telugu-{lora,qlora}` every 250 steps.

## 1. Parameters

In [ ]:
RUN = "lora"      # "lora" or "qlora"
RESUME = False    # True only to continue a killed run from the Hub's last-checkpoint/

## 2. Secrets, single GPU, environment

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

assert RUN in ("lora", "qlora"), RUN
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PATH"] = "/tmp/axo-venv/bin:" + os.environ["PATH"]
print("RUN =", RUN, "| RESUME =", RESUME, "| HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## 3. Install Axolotl (pinned, isolated venv) and print versions

In [ ]:
%%bash
set -e
pip install -q uv
uv venv --python 3.12 --allow-existing /tmp/axo-venv
PY=/tmp/axo-venv/bin/python
uv pip install --python $PY -q packaging setuptools wheel ninja
uv pip install --python $PY -q --torch-backend auto "torch==2.13.0"
uv pip install --python $PY -q --torch-backend auto --no-build-isolation "axolotl==0.20.0"
$PY -c "import torch, transformers, peft, bitsandbytes, axolotl; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__, axolotl.__version__)"

## 4. Get the code

In [ ]:
%%bash
rm -rf /kaggle/working/telugulora
git clone --depth 1 https://github.com/pashamgithub/telugulora.git /kaggle/working/telugulora
cd /kaggle/working/telugulora && git log --oneline -1

## 5. (Only if RESUME) download the last checkpoint from the Hub

In [ ]:
resume_arg = ""
if RESUME:
    from huggingface_hub import snapshot_download
    out_dir = f"/kaggle/working/outputs/{RUN}"
    snapshot_download(f"thirumalreddy0172/qwen3-4b-telugu-{RUN}", allow_patterns="last-checkpoint/*", local_dir=out_dir)
    resume_arg = f"--resume-from-checkpoint {out_dir}/last-checkpoint"
print("resume_arg:", resume_arg or "(fresh run)")

## 6. Train (~6–7 h)

In [ ]:
import shlex, subprocess

os.chdir("/kaggle/working/telugulora")
cmd = ["axolotl", "train", f"configs/qwen3-4b-{RUN}.yml", *shlex.split(resume_arg), "--", "--num_processes", "1"]
print("Running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
assert proc.wait() == 0, f"training failed (exit code {proc.returncode})"


## 7. Confirm the upload

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files(f"thirumalreddy0172/qwen3-4b-telugu-{RUN}")
for f in files:
    if not f.startswith("last-checkpoint/"):
        print(f)
print("TensorBoard logs on Hub:", any(f.startswith("runs/") for f in files))